# 01 — Exploratory Data Analysis & Randomization Check

The target is treatment-effect heterogeneity, not conversion probability: an uplift model, not a propensity-to-buy classifier. Targeting on P(convert) rewards customers who'd buy anyway; the question here is who to email to maximize the incremental effect of the campaign.

The Hillstrom dataset (`Womens E-Mail` / `Mens E-Mail` / `No E-Mail`) is a real randomized experiment, which licenses a causal reading of later effect estimates — conditional on randomization actually holding and the covariate set being well-behaved. Both are checked below before any modeling.

## Step 1: Load data and quality checks

Data is loaded through `src/data_prep.load_hillstrom`, the single shared entry point used
by every notebook and the Streamlit app, so column typing never drifts between them. The
raw CSV is downloaded automatically on first use if `data/raw/hillstrom.csv` is missing.

In [1]:
%pip install pandas numpy matplotlib plotly scikit-uplift scikit-learn statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

# Make src/ importable regardless of whether Jupyter was launched from the
# project root or from inside notebooks/.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from src.data_prep import (
    load_hillstrom,
    compute_balance_table,
    check_history_segment_consistency,
    check_design_matrix_rank,
    compute_vif,
)

pd.set_option("display.max_columns", None)

In [3]:
df = load_hillstrom("../data/raw/hillstrom.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.440002,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.079987,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.649994,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.830017,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.340000,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [4]:
# Missingness is verified rather than assumed — silent missing values would
# bias both outcome rates and the balance/structural checks below.
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
})
missing

,n_missing,pct_missing
recency,0,0.0
history_segment,0,0.0
history,0,0.0
mens,0,0.0
womens,0,0.0
zip_code,0,0.0
newbie,0,0.0
channel,0,0.0
segment,0,0.0
visit,0,0.0


In [5]:
# Verify dtypes actually match what data_prep.py's load_hillstrom promises:
# category for zip_code/channel/segment, ordered category for history_segment,
# int8 for binary flags, downcast float for history/spend.
df.dtypes

recency                int8
history_segment    category
history             float32
mens                   int8
womens                 int8
zip_code           category
newbie                 int8
channel            category
segment            category
visit                  int8
conversion             int8
spend               float32
dtype: object

## Step 2: Univariate exploration

A quick look at every variable in isolation — covariates and outcomes alike — before any
balance or structural check. This is what catches skew, rare categories, or a zero-inflated
outcome that should shape evaluation choices later (notebook 04).

In [6]:
# Numeric covariates: recency, history
display(df[["recency", "history"]].describe())

px.histogram(df, x="recency", nbins=12, title="Recency (months since last purchase)").show()
px.histogram(df, x="history", nbins=60, title="History (past year's spend, $)").show()
print(f"Skewness of history: {df['history'].skew():.2f}")

,recency,history
count,64000.000000,64000.000000
mean,5.763734,242.085663
std,3.507592,256.158600
min,1.000000,29.990000
25%,2.000000,64.660004
50%,6.000000,158.110001
75%,9.000000,325.657494
max,12.000000,3345.929932


Skewness of history: 2.42


In [7]:
# history_segment is an ordered categorical (see data_prep.py), so this shows
# bins in their natural $0 -> $1,000+ order rather than alphabetically.
counts = df["history_segment"].value_counts().reindex(df["history_segment"].cat.categories)
px.bar(counts, title="history_segment distribution (bin order preserved)").update_layout(
    showlegend=False
).show()

In [8]:
# mens/womens are independent purchase-history flags, not a mutually
# exclusive gender label — a customer can have both set to 1.
for col in ["mens", "womens", "newbie"]:
    print(df[col].value_counts(normalize=True).rename(f"{col} (proportion)"), "\n")

crosstab = pd.crosstab(df["mens"], df["womens"], normalize=True) * 100
px.imshow(crosstab, text_auto=".1f", title="mens vs womens co-occurrence (% of customers)").show()

mens
1    0.551031
0    0.448969
Name: mens (proportion), dtype: float64 

womens
1    0.549719
0    0.450281
Name: womens (proportion), dtype: float64 

newbie
1    0.50225
0    0.49775
Name: newbie (proportion), dtype: float64 



In [9]:
# Categorical covariates
for col in ["zip_code", "channel"]:
    px.bar(df[col].value_counts(), title=f"{col} distribution").update_layout(
        showlegend=False
    ).show()

In [10]:
# Treatment assignment — sanity-checks arm sizes; the actual covariate
# balance check is Step 3.
px.bar(df["segment"].value_counts(), title="Customers per segment (treatment arm)").update_layout(
    showlegend=False
).show()

In [11]:
# Outcomes. spend is expected to be strongly zero-inflated: only converters
# have nonzero spend, and conversion is rare.
for col in ["visit", "conversion"]:
    print(df[col].value_counts(normalize=True).rename(f"{col} (proportion)"), "\n")

pct_zero_spend = (df["spend"] == 0).mean() * 100
print(f"% of customers with spend == 0: {pct_zero_spend:.1f}%")
px.histogram(df.loc[df["spend"] > 0, "spend"], nbins=50, title="spend, among spend > 0").show()

visit
0    0.853219
1    0.146781
Name: visit (proportion), dtype: float64 

conversion
0    0.990969
1    0.009031
Name: conversion (proportion), dtype: float64 

% of customers with spend == 0: 99.1%


## Step 3: Randomization balance check

Random assignment is what licenses a causal reading of any outcome difference we observe
later: under randomization, **ignorability** holds by design (treatment is independent of
potential outcomes), so no adjustment for confounding is needed. But "randomized" is a
design property, not a guarantee the *realized* sample stayed balanced — we check that
pre-treatment covariates are actually similar across the three arms.

We use the **standardized mean difference (SMD)** rather than raw mean differences, since it
scales by pooled variability and is therefore comparable across covariates in different
units. `|SMD| > 0.1` is the conventional flag threshold from the matching literature (not a
formal test). We also rely on **SUTVA**: one customer's assignment doesn't affect another's
outcome — plausible here since an email to one customer shouldn't change another's behavior,
but worth stating since every causal claim in this project rests on it.

`history_segment` is excluded below since it duplicates `history`'s information (checked
structurally in Step 4).

In [12]:
covariates = ["recency", "history", "mens", "womens", "zip_code", "newbie", "channel"]

balance_table = compute_balance_table(
    df, covariates=covariates, treatment_col="segment", reference_group="No E-Mail",
)
display(balance_table.reindex(balance_table["smd"].abs().sort_values(ascending=False).index))

flagged = balance_table[balance_table["flag_imbalanced"]]
if flagged.empty:
    print("No covariate/group combinations flagged as imbalanced (|SMD| > 0.1).")
else:
    print(f"{len(flagged)} covariate/group combination(s) flagged as imbalanced:")
    display(flagged)

,covariate,group,reference,smd,flag_imbalanced
13,zip_code_Rural,Mens E-Mail,No E-Mail,0.013659,False
15,zip_code_Surburban,Mens E-Mail,No E-Mail,-0.011743,False
27,channel_Web,Mens E-Mail,No E-Mail,0.011014,False
6,mens_1,Womens E-Mail,No E-Mail,-0.008631,False
4,mens_0,Womens E-Mail,No E-Mail,0.008631,False
24,channel_Phone,Womens E-Mail,No E-Mail,0.008623,False
25,channel_Phone,Mens E-Mail,No E-Mail,-0.008276,False
3,history,Mens E-Mail,No E-Mail,0.007613,False
9,womens_0,Mens E-Mail,No E-Mail,-0.007589,False
11,womens_1,Mens E-Mail,No E-Mail,0.007589,False


No covariate/group combinations flagged as imbalanced (|SMD| > 0.1).


All 28 covariate/group combinations have |SMD| well under the 0.1 threshold — the largest is 0.014 (`zip_code_Rural`, Mens E-Mail vs. control), and none are flagged. Randomization held in the realized sample.

## Step 4: Structural checks — is the covariate set full rank?

Independent of balance, the covariate *set itself* can be structurally redundant in ways
that only matter for certain model classes:

- **`history_segment` vs `history`** — `history_segment` bins the continuous `history`
  column, so the two carry almost the same information. Harmless for tree-based learners
  (scikit-uplift's uplift trees, most meta-learners used later), but a real concern for a
  linear/logistic base learner, where redundant features inflate coefficient variance without
  adding signal. Takeaway: never feed both into the same linear base learner.
- **The dummy variable trap** — one-hot encoding a $k$-level categorical with an intercept
  present makes the design matrix rank-deficient by construction (the $k$ dummies sum to 1,
  duplicating the intercept column). Fixed by `drop_first=True`; checked explicitly below
  rather than assumed.

We also compute **VIF** (Variance Inflation Factor) per encoded covariate, which catches
*near*-collinearity that an exact rank check would miss: VIF > 5 is conventionally read as
moderate multicollinearity, VIF > 10 as severe — actionable only if a linear/logistic base
learner is used.

In [13]:
# Is history_segment a clean, non-overlapping binning of history?
check_history_segment_consistency(df)

,history_segment,n,history_min,history_max
0,1) $0 - $100,22970,29.990000,99.989998
1,2) $100 - $200,14254,100.000000,199.979996
2,3) $200 - $350,12289,200.000000,349.959991
3,4) $350 - $500,6409,350.010010,499.950012
4,5) $500 - $750,4911,500.000000,749.789978
5,"6) $750 - $1,000",1859,750.010010,999.750000
6,"7) $1,000 +",1308,1000.150024,3345.929932


In [14]:
numeric_covariates = ["recency", "history"]
categorical_covariates = ["zip_code", "channel"]

corr = df[["recency", "history", "mens", "womens", "newbie"]].corr()
px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu", zmin=-1, zmax=1,
          title="Correlation matrix — numeric/binary covariates").show()

rank_with_trap = check_design_matrix_rank(
    df, numeric_covariates, categorical_covariates, drop_first=False, include_intercept=True
)
rank_fixed = check_design_matrix_rank(
    df, numeric_covariates, categorical_covariates, drop_first=True, include_intercept=True
)
print("With intercept, drop_first=False (dummy variable trap):", rank_with_trap)
print("With intercept, drop_first=True  (correctly specified):  ", rank_fixed)

compute_vif(df, numeric_covariates, categorical_covariates)

With intercept, drop_first=False (dummy variable trap): {'n_columns': 9, 'rank': 7, 'full_rank': False}
With intercept, drop_first=True  (correctly specified):   {'n_columns': 7, 'rank': 7, 'full_rank': True}


,covariate,vif
0,channel_Phone,3.045778
1,channel_Web,3.042093
2,zip_code_Surburban,2.206659
3,zip_code_Urban,2.206596
4,history,1.258941
5,recency,1.064878


With `zip_code` and `channel` both included, `rank_with_trap` is 2 columns short of `n_columns` (9 vs. 7) — one redundant dummy per categorical, confirming the trap compounds once per categorical variable rather than by a fixed offset. `rank_fixed` is full rank (7/7), confirming `drop_first=True` fixes it. VIF tops out at 3.05 (`channel_Phone`), comfortably under the 5 threshold — no linear base learner needs extra collinearity treatment.

## Step 5: Outcome rates by segment (sanity check)

Purely descriptive — not yet heterogeneous treatment effects, which is the goal of later
notebooks — but a useful check that the campaign has *some* aggregate effect, and that both
email arms outperform the no-email control in the expected direction before investing in
uplift models.

In [15]:
rates = (
    df.groupby("segment", observed=True)[["visit", "conversion"]]
    .mean().reset_index()
    .melt(id_vars="segment", var_name="outcome", value_name="rate")
)
px.bar(rates, x="segment", y="rate", color="outcome", barmode="group",
       title="Visit and Conversion Rates by Segment").show()

spend_by_segment = df.groupby("segment", observed=True)["spend"].mean().reset_index()
px.bar(spend_by_segment, x="segment", y="spend", title="Average Spend by Segment").show()

## Interpretation (for `docs/methodology.md`)

Across the pre-treatment covariates checked (recency, history, purchase-category flags, newbie status, zip code region, channel), standardized mean differences between each email segment and the no-email control were all well below the 0.1 threshold (max 0.014, `zip_code_Rural` for Mens E-Mail), with zero flagged combinations — consistent with randomization holding as designed. Structurally, `history_segment` is a clean, non-overlapping binning of `history`, so the two are never used together in a linear base learner; the one-hot encoded covariate set is full rank once the dummy variable trap is avoided (`drop_first=True`), and VIF tops out at 3.05, well under the 5 threshold, so no collinearity adjustment is needed. Raw outcome rates confirm both email arms beat the no-email control: visit rate 18.3% (Mens) / 15.1% (Womens) vs. 10.6% (control); conversion 1.25% / 0.88% vs. 0.57%; average spend $1.42 / $1.08 vs. $0.65. Randomization/ignorability and SUTVA hold, and the covariate structure is sound — proceeding to uplift modeling with reasonable confidence that estimated effects reflect the campaign's causal impact rather than confounding or modeling artifacts.